In [1]:
# 2D U(1) Villain TN test on a 2x2 periodic lattice
import numpy as np
import math

# -----------------------------
# Parameters
# -----------------------------
beta = 1.0       # inverse coupling in Villain weight
Nmax = 2         # flux cutoff |n_p| <= Nmax
# (2*Nmax+1)^4 configurations for 4 plaquettes

# -----------------------------
# Plaquette indexing and lattice geometry
# -----------------------------
# We have a 2x2 lattice of plaquettes with periodic BCs:
# plaquettes P(i_p, j_p), i_p, j_p ∈ {0,1}
# We'll map them to indices 0..3: id(i_p, j_p) = i_p + 2 * j_p

def plaq_id(i_p, j_p):
    return (i_p % 2) + 2 * (j_p % 2)

# Sites S(i,j), i,j ∈ {0,1}, also periodic.
# At each site, four plaquettes meet:
# We'll use the pattern (NW, NE, SE, SW) around the site.
# Bianchi constraint at a site: n_NW - n_NE + n_SE - n_SW = 0.

def star_plaquettes(i, j):
    """
    Return the 4 plaquette indices (NW, NE, SE, SW)
    around site (i,j) on the 2x2 periodic lattice.
    """
    # periodic shifts
    NW = plaq_id(i - 1, j - 1)
    NE = plaq_id(i,     j - 1)
    SE = plaq_id(i,     j)
    SW = plaq_id(i - 1, j)
    return (NW, NE, SE, SW)

# Precompute star mappings for the 4 sites:
# Site order: S0=(0,0), S1=(1,0), S2=(0,1), S3=(1,1)
site_coords = [(0,0), (1,0), (0,1), (1,1)]
site_stars = [star_plaquettes(i,j) for (i,j) in site_coords]

print("Site stars (NW,NE,SE,SW) -> plaquette indices 0..3:")
for idx, sc in enumerate(site_coords):
    print(f"  Site {idx} at {sc}: star={site_stars[idx]}")


# -----------------------------
# Exact flux sum at θ = 0
# -----------------------------
def Z_flux_exact(beta, Nmax):
    """
    Direct sum over plaquette fluxes n_p on a 2x2 lattice,
    enforcing Bianchi constraints at each site, with Villain weight:
      weight = exp(-2π^2 β Σ_p n_p^2)
    """
    Z = 0.0
    # All possible flux assignments n_p ∈ [-Nmax, ..., Nmax] for 4 plaquettes
    vals = np.arange(-Nmax, Nmax + 1)
    for n0 in vals:
        for n1 in vals:
            for n2 in vals:
                for n3 in vals:
                    n = np.array([n0, n1, n2, n3], dtype=np.int64)

                    # Bianchi constraints at each of the 4 sites
                    ok = True
                    for star in site_stars:
                        # order is (NW, NE, SE, SW)
                        n_NW = n[star[0]]
                        n_NE = n[star[1]]
                        n_SE = n[star[2]]
                        n_SW = n[star[3]]
                        if (n_NW - n_NE + n_SE - n_SW) != 0:
                            ok = False
                            break
                    if not ok:
                        continue

                    # Villain weight: product over plaquettes of exp(-2π^2 β n_p^2)
                    # which is exp(-2π^2 β Σ_p n_p^2)
                    exponent = -2.0 * (math.pi**2) * beta * np.sum(n**2)
                    weight = math.exp(exponent)
                    Z += weight
    return Z

# -----------------------------
# Site tensor T^{(x)}_{n_NW, n_NE, n_SE, n_SW}
# -----------------------------
def build_site_tensor(beta, Nmax):
    """
    Build the 4-index site tensor T with axes (NW, NE, SE, SW),
    entries:
        T[n1,n2,n3,n4] = δ_{n1 - n2 + n3 - n4, 0}
                         * Π_j exp(-(2π^2 β / 4) * n_j^2)
    where each plaquette flux gets its weight split across 4 sites.
    """
    vals = np.arange(-Nmax, Nmax + 1)
    D = vals.size
    T = np.zeros((D, D, D, D), dtype=np.float64)

    # Precompute local factor exp(-2π^2 β n^2 / 4) for each n
    local_weight = np.exp(-2.0 * (math.pi**2) * beta * (vals**2) / 4.0)

    for i1, n1 in enumerate(vals):
        for i2, n2 in enumerate(vals):
            for i3, n3 in enumerate(vals):
                for i4, n4 in enumerate(vals):
                    # Bianchi constraint at site: n_NW - n_NE + n_SE - n_SW = 0
                    if (n1 - n2 + n3 - n4) != 0:
                        continue
                    # Weight split: product of 4 local factors
                    w = local_weight[i1] * local_weight[i2] * local_weight[i3] * local_weight[i4]
                    T[i1, i2, i3, i4] = w

    return T

# -----------------------------
# TN contraction on 2x2 lattice
# -----------------------------
def Z_TN(beta, Nmax):
    """
    Contract the 4-site tensor network on a 2x2 periodic lattice.

    We have 4 plaquette indices: P0, P1, P2, P3.
    We'll label them 'a','b','c','d' respectively.

    Each site tensor T_site has axes (NW,NE,SE,SW),
    and each axis is identified with one of {a,b,c,d} depending on star_plaquettes.
    The global contraction is an einsum over all shared plaquette indices.
    """
    T_site = build_site_tensor(beta, Nmax)
    D = T_site.shape[0]

    # Just to confirm D matches flux range
    assert D == 2 * Nmax + 1

    # We map plaquettes 0,1,2,3 -> labels 'a','b','c','d'.
    # Using the precomputed stars for the 4 sites, we know:
    # Site 0 star: (NW,NE,SE,SW) -> (3,2,0,1) -> (d,c,a,b)
    # Site 1 star: (2,3,1,0) -> (c,d,b,a)
    # Site 2 star: (1,0,2,3) -> (b,a,c,d)
    # Site 3 star: (0,1,3,2) -> (a,b,d,c)
    # We'll use einsum with appropriately ordered index labels.

    Z = np.einsum(
        'dcab,cdba,bacd,abdc->',
        T_site, T_site, T_site, T_site
    )
    return Z

# -----------------------------
# Run the comparison
# -----------------------------
Z_exact = Z_flux_exact(beta, Nmax)
Z_tn = Z_TN(beta, Nmax)

print(f"beta={beta}, Nmax={Nmax}")
print(f"Z_exact (flux sum)      = {Z_exact:.16e}")
print(f"Z_TN    (tensor contr.) = {Z_tn:.16e}")
print(f"Absolute difference     = {abs(Z_exact - Z_tn):.3e}")
print(f"Relative difference     = {abs(Z_exact - Z_tn)/Z_exact:.3e}")


Site stars (NW,NE,SE,SW) -> plaquette indices 0..3:
  Site 0 at (0, 0): star=(3, 2, 0, 1)
  Site 1 at (1, 0): star=(2, 3, 1, 0)
  Site 2 at (0, 1): star=(1, 0, 2, 3)
  Site 3 at (1, 1): star=(0, 1, 3, 2)
beta=1.0, Nmax=2
Z_exact (flux sum)      = 1.0000000000000000e+00
Z_TN    (tensor contr.) = 1.0000000000000000e+00
Absolute difference     = 0.000e+00
Relative difference     = 0.000e+00
